# Craig Pricing Service — Demo Notebook
Run each cell to see how the quoting engine works with real examples from Justin's pricing sheets.

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from fastapi.testclient import TestClient
from main import app
import json

client = TestClient(app)

def quote(endpoint, payload):
    """Helper — sends a quote request and pretty-prints the response."""
    r = client.post(endpoint, json=payload)
    data = r.json()
    print(json.dumps(data, indent=2))
    return data

print('Ready.')

---
## 1. Simple Quote — 500 A5 Flyers, single-sided, gloss
Straight lookup. Sheet says €22.00 for 500 A5 flyers.

In [ ]:
quote("/quote/small-format", {
    "product": "flyers_a5",
    "quantity": 500,
    "double_sided": False,
    "finish": "gloss"
})

---
## 2. Double-Sided Surcharge — 500 A5 Flyers, double-sided
Same product but double-sided → base €22.00 × 1.20 = **€26.40**

In [ ]:
quote("/quote/small-format", {
    "product": "flyers_a5",
    "quantity": 500,
    "double_sided": True,
    "finish": "gloss"
})

---
## 3. Business Cards — Double-sided is FREE (exception)
Business cards don't charge extra for double-sided. 500 cards = €38 either way.

In [ ]:
# Single-sided
print('=== Single-sided ===')
quote("/quote/small-format", {
    "product": "business_cards",
    "quantity": 500,
    "double_sided": False,
    "finish": "gloss"
})

print('\n=== Double-sided (same price!) ===')
quote("/quote/small-format", {
    "product": "business_cards",
    "quantity": 500,
    "double_sided": True,
    "finish": "gloss"
})

---
## 4. Soft-Touch Finish — 250 Business Cards
Soft-touch = +25% across the board. Base €60.00 × 1.25 = **€75.00**

In [ ]:
quote("/quote/small-format", {
    "product": "business_cards",
    "quantity": 250,
    "double_sided": True,
    "finish": "soft-touch"
})

---
## 5. Both Surcharges Stacked — 1,000 A4 Flyers, double-sided + soft-touch
Base €22.00 × 1.20 (double) × 1.25 (soft-touch) = **€33.00**

In [ ]:
quote("/quote/small-format", {
    "product": "flyers_a4",
    "quantity": 1000,
    "double_sided": True,
    "finish": "soft-touch"
})

---
## 6. NCR Pads — Triplicate (+10%)
20 A4 NCR pads, triplicate. Base €85.00 × 1.10 = **€93.50**

In [ ]:
quote("/quote/small-format", {
    "product": "ncr_pads_a4",
    "quantity": 20,
    "double_sided": False,
    "finish": "triplicate"
})

---
## 7. With Artwork — 250 Business Cards + 2 hours design
Print: €60.00 + VAT = €73.80 | Artwork: 2h × €65 = €130.00 + VAT = €159.90 | **Total: €233.70**

In [ ]:
quote("/quote/small-format", {
    "product": "business_cards",
    "quantity": 250,
    "double_sided": False,
    "finish": "gloss",
    "needs_artwork": True,
    "artwork_hours": 2
})

---
## 8. Large Format — Roller Banners (unit vs bulk)
2 banners = €120 each. 5+ banners = €110 each (bulk).

In [ ]:
print('=== 2 banners (standard price) ===')
quote("/quote/large-format", {
    "product": "roller_banners",
    "quantity": 2
})

print('\n=== 5 banners (bulk kicks in) ===')
quote("/quote/large-format", {
    "product": "roller_banners",
    "quantity": 5
})

---
## 9. Large Format — PVC Banners (per sq/m)
8 sq/m at €28/sqm = €224. 12 sq/m at bulk €23/sqm = €276.

In [ ]:
print('=== 8 sq/m (standard) ===')
quote("/quote/large-format", {
    "product": "pvc_banners",
    "quantity": 8
})

print('\n=== 12 sq/m (bulk >= 10) ===')
quote("/quote/large-format", {
    "product": "pvc_banners",
    "quantity": 12
})

---
## 10. Booklet Quote — A5 Saddle Stitch, 24pp, Card Cover + Lam, 250 copies
Direct lookup from booklet sheet = **€697.00**

In [ ]:
quote("/quote/booklet", {
    "format": "a5",
    "binding": "saddle_stitch",
    "pages": 24,
    "cover_type": "card_cover_lam",
    "quantity": 250
})

---
## 11. Booklet — A4 Perfect Bound, 64pp, Card Cover, 100 copies
Sheet says **€948.00**

In [ ]:
quote("/quote/booklet", {
    "format": "a4",
    "binding": "perfect_bound",
    "pages": 64,
    "cover_type": "card_cover",
    "quantity": 100
})

---
## 12. ESCALATION — Quantity not on sheet
750 A5 flyers doesn't exist on the pricing sheet → Craig escalates to Justin.

In [ ]:
quote("/quote/small-format", {
    "product": "flyers_a5",
    "quantity": 750,
    "double_sided": False,
    "finish": "gloss"
})

---
## 13. ESCALATION — Invalid finish for product
Letterheads only come in uncoated. Asking for gloss → escalate.

In [ ]:
quote("/quote/small-format", {
    "product": "letterheads",
    "quantity": 250,
    "double_sided": False,
    "finish": "gloss"
})

---
## 14. ESCALATION — Booklet page count not on sheet
10pp saddle stitch doesn't exist → escalate with available page counts.

In [ ]:
quote("/quote/booklet", {
    "format": "a5",
    "binding": "saddle_stitch",
    "pages": 10,
    "cover_type": "self_cover",
    "quantity": 100
})

---
## 15. Browse the full product catalog

In [ ]:
r = client.get("/products")
data = r.json()

for category in data:
    print(f"\n{'='*60}")
    print(f"  {category['category'].upper()} — {len(category['products'])} products")
    print(f"{'='*60}")
    for p in category['products'][:5]:  # Show first 5 per category
        print(f"  • {p.get('name') or p.get('format', '') + ' ' + p.get('binding', '')}")
    if len(category['products']) > 5:
        print(f"  ... and {len(category['products']) - 5} more")